# Módulo 13 · Práctica: atención desde cero

En este notebook programas con NumPy el mecanismo que está dentro de Claude, Gemini o Grok: la atención por producto escalar escalado. Empiezas con una cabeza, añades la máscara causal, construyes la versión multi-cabeza y la comparas con la implementación oficial de PyTorch. Al final dibujas mapas de atención y montas a mano una cabeza de inducción.

**Cómo trabajar:** ejecuta las celdas en orden. Las celdas `# TU CÓDIGO` son ejercicios; la celda `# Comprobación` de después te dice si están bien.

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación: softmax estable |
| 1 | Atención de una cabeza y el ejemplo de 3 tokens |
| 2 | Por qué se divide entre $\sqrt{d_k}$: simulación |
| 3 | Máscara causal |
| 4 | Atención multi-cabeza |
| 5 | Verificación contra PyTorch |
| 6 | Mapas de atención y cabeza de inducción |
| 7 | El coste cuadrático, medido |

## Parte 0 · Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.set_printoptions(precision=3, suppress=True)

def softmax(z, axis=-1):
    # Versión estable (módulo 11): restamos el máximo de cada fila.
    # Funciona con -inf siempre que cada fila tenga al menos un valor finito.
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

print(softmax(np.array([2.0, 0.0, 2.0])))
print(softmax(np.array([0.0, 4.0, -np.inf])))

## Parte 1 · Atención de una cabeza

$$\mathrm{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Q y K tienen forma `(n, d_k)` y V forma `(n, d_v)`. Usa `@` para el producto matricial y `K.T` para la traspuesta.

**Ejercicio 1.** Implementa `atencion(Q, K, V)`. Debe devolver la salida `O` (forma `(n, d_v)`) y el mapa de atención `A` (forma `(n, n)`).

In [ ]:
def atencion(Q, K, V):
    # TU CÓDIGO
    A = None
    O = None
    return O, A

Comprobamos con el ejemplo resuelto a mano en la sección 13.2 del documento.

In [ ]:
# Comprobación
Q = np.array([[2, 0, 2, 0], [0, 2, 0, 2], [2, 2, 0, 0]], dtype=float)
K = np.array([[2, 0, 0, 0], [0, 2, 0, 2], [1, 1, 1, 1]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
O, A = atencion(Q, K, V)
assert A is not None and A.shape == (3, 3), "A debe ser 3x3"
assert np.allclose(A.sum(axis=1), 1), "Cada fila de A debe sumar 1"
assert np.allclose(A[1], [0.0159, 0.8668, 0.1173], atol=1e-3), "Revisa el escalado entre sqrt(d_k)"
assert np.allclose(O, [[0.9366, 0.5317], [0.1332, 0.9841], [0.6667, 0.6667]], atol=1e-3), "Revisa O = A @ V"
print("Correcto\nA =\n", A, "\nO =\n", O)

## Parte 2 · Por qué se divide entre $\sqrt{d_k}$

En el documento demostraste que, si las componentes de $q$ y $k$ son independientes con media 0 y varianza 1, entonces $\mathrm{Var}(q\cdot k) = d_k$. Vamos a comprobarlo con Monte Carlo (módulo 5).

**Ejercicio 2.** Escribe `varianza_producto(d_k, m, rng)`: genera `m` pares de vectores gaussianos estándar de dimensión `d_k`, calcula sus `m` productos escalares y devuelve su varianza empírica. Pista: `rng.standard_normal((m, d_k))` y `(q * k).sum(axis=1)`.

In [ ]:
def varianza_producto(d_k, m, rng):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
rng = np.random.default_rng(0)
dims = [1, 4, 16, 64, 256]
vars_ = [varianza_producto(d, 50_000, rng) for d in dims]
for d, v in zip(dims, vars_):
    print(f"d_k = {d:>3}: Var(q·k) = {v:8.2f}   Var(q·k/√d_k) = {v / d:.3f}")
assert all(abs(v / d - 1) < 0.05 for d, v in zip(dims, vars_)), "La varianza debería ser ≈ d_k"
print("Correcto")

Ahora mira qué le pasa a la softmax si no escalamos: con $d_k$ grande la distribución se vuelve casi *one-hot* (saturada) y su gradiente se anula.

In [ ]:
rng = np.random.default_rng(1)
n = 8
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, d_k in zip(axes, [4, 256]):
    q = rng.standard_normal(d_k)
    Kr = rng.standard_normal((n, d_k))
    s = Kr @ q
    ax.bar(np.arange(n) - 0.2, softmax(s), width=0.4, label="sin escalar")
    ax.bar(np.arange(n) + 0.2, softmax(s / np.sqrt(d_k)), width=0.4, label="÷ √d_k")
    ax.set_title(f"d_k = {d_k}"); ax.set_xlabel("key j")
axes[0].set_ylabel("peso α_j"); axes[0].legend()
plt.tight_layout(); plt.show()

## Parte 3 · Máscara causal

Un LLM no puede mirar al futuro. Sumamos $-\infty$ a las posiciones $j > i$ **antes** de la softmax:

$$M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i\end{cases}$$

**Ejercicio 3.** (a) Escribe `mascara_causal(n)` que devuelva la matriz $M$ de forma `(n, n)`. Pista: `np.triu(..., k=1)` selecciona lo que está por encima de la diagonal. (b) Escribe `atencion_causal(Q, K, V)` que sume la máscara antes de la softmax.

In [ ]:
def mascara_causal(n):
    # TU CÓDIGO
    return None

def atencion_causal(Q, K, V):
    # TU CÓDIGO
    A = None
    O = None
    return O, A

In [ ]:
# Comprobación
M = mascara_causal(3)
assert M is not None and M.shape == (3, 3)
assert np.all(M[np.tril_indices(3)] == 0) and np.all(np.isneginf(M[np.triu_indices(3, k=1)])), "0 en y bajo la diagonal, -inf encima"
O, A = atencion_causal(Q, K, V)
assert np.allclose(np.triu(A, k=1), 0), "Los pesos futuros deben ser exactamente 0"
assert np.allclose(A.sum(axis=1), 1), "Las filas deben seguir sumando 1"
assert np.allclose(A[1, :2], [0.018, 0.982], atol=1e-3), "Fila 2 del ejemplo: (0,018; 0,982; 0)"
# Causalidad: cambiar el último token no puede cambiar las salidas anteriores
rng = np.random.default_rng(2)
Xq, Xk, Xv = rng.standard_normal((3, 6, 4))
O1, _ = atencion_causal(Xq, Xk, Xv)
Xk2, Xv2 = Xk.copy(), Xv.copy(); Xk2[-1] += 5; Xv2[-1] -= 5
O2, _ = atencion_causal(Xq, Xk2, Xv2)
assert np.allclose(O1[:-1], O2[:-1]), "Las salidas 0..n-2 no deben depender del último token"
print("Correcto\nA causal =\n", A)

## Parte 4 · Atención multi-cabeza

Con $h$ cabezas y dimensión del modelo $d$, cada cabeza usa $d_k = d/h$. El truco eficiente: calcular $Q = XW_Q$ entero y **reordenar** de `(n, d)` a `(h, n, d/h)`. Así la atención de todas las cabezas se calcula de una vez gracias al *broadcasting*.

```
(n, d) --reshape--> (n, h, d_k) --transpose(1, 0, 2)--> (h, n, d_k)
```

**Ejercicio 4.** (a) `dividir_cabezas(X, h)`: de `(n, d)` a `(h, n, d/h)`. (b) `unir_cabezas(X)`: la operación inversa. (c) `multihead(X, WQ, WK, WV, WO, h, causal=True)`: proyecta, divide, aplica la atención (causal o no) a todas las cabezas a la vez, une y multiplica por `WO`. Devuelve la salida `(n, d)` y los mapas `(h, n, n)`.

Tu `atencion_causal` ya funciona con un eje extra delante si usaste `np.swapaxes(K, -1, -2)` en lugar de `K.T`. Si no, corrígela.

In [ ]:
def dividir_cabezas(X, h):
    # TU CÓDIGO
    return None

def unir_cabezas(X):
    # TU CÓDIGO
    return None

def multihead(X, WQ, WK, WV, WO, h, causal=True):
    # TU CÓDIGO
    Y, A = None, None
    return Y, A

In [ ]:
# Comprobación
rng = np.random.default_rng(3)
n, d, h = 5, 8, 2
X = rng.standard_normal((n, d))
WQ, WK, WV, WO = (rng.standard_normal((d, d)) / np.sqrt(d) for _ in range(4))
assert dividir_cabezas(X, h).shape == (h, n, d // h), "Forma esperada (h, n, d/h)"
assert np.allclose(unir_cabezas(dividir_cabezas(X, h)), X), "unir_cabezas debe deshacer dividir_cabezas"
Y, A = multihead(X, WQ, WK, WV, WO, h)
assert Y.shape == (n, d) and A.shape == (h, n, n)
# Referencia lenta: cabeza a cabeza con un bucle
dk = d // h
cabezas = []
for i in range(h):
    sl = slice(i * dk, (i + 1) * dk)
    Oi, _ = atencion_causal(X @ WQ[:, sl], X @ WK[:, sl], X @ WV[:, sl])
    cabezas.append(Oi)
Y_ref = np.concatenate(cabezas, axis=1) @ WO
assert np.allclose(Y, Y_ref), "La versión vectorizada debe coincidir con el bucle por cabezas"
print("Correcto. Parámetros de la capa (4·d²):", sum(W.size for W in (WQ, WK, WV, WO)))

## Parte 5 · Verificación contra PyTorch

`torch.nn.functional.scaled_dot_product_attention` es la versión oficial (en GPU usa núcleos como FlashAttention, módulo 17). Si PyTorch está instalado, comparamos; si no, la celda lo indica y sigue.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    TORCH = True
except ImportError:
    TORCH = False
    print("PyTorch no está instalado: se omite esta comparación.")

if TORCH:
    rng = np.random.default_rng(4)
    h, n, dk = 4, 7, 16
    Qn, Kn, Vn = rng.standard_normal((3, h, n, dk))
    Qt, Kt, Vt = (torch.tensor(a, dtype=torch.float64) for a in (Qn, Kn, Vn))
    for causal in (False, True):
        ref = F.scaled_dot_product_attention(Qt, Kt, Vt, is_causal=causal).numpy()
        if causal:
            mio, _ = atencion_causal(Qn, Kn, Vn)
        else:
            mio = softmax(Qn @ np.swapaxes(Kn, -1, -2) / np.sqrt(dk)) @ Vn
        err = np.abs(mio - ref).max()
        print(f"causal={causal}: error máximo = {err:.2e}")
        assert err < 1e-10, "Tu atención no coincide con la de PyTorch"
    print("Correcto: coincide con PyTorch")

## Parte 6 · Mapas de atención y cabeza de inducción

Un mapa de atención es la matriz $A$ dibujada como imagen: fila = token que mira, columna = token mirado. Vamos a construir **a mano** las dos cabezas del circuito de inducción (sección 13.7) para la frase de abajo, con embeddings *one-hot* y un factor $\beta = 10$ que hace la softmax casi dura.

- **Capa 1, cabeza del token anterior.** Query = posición $i$; key = posición $j+1$. La posición $i$ encaja con $j = i-1$ y copia su token: $h_i \approx$ token anterior.
- **Capa 2, cabeza de inducción.** Query = token actual; key = $h_j$ («mi anterior era…»); value = token en $j$.

Usamos `atencion_causal` con `Q` y `K` multiplicados por $\beta$: los parecidos crecen como $\beta^2/\sqrt{d_k}$ y la softmax se vuelve casi un argmax.

In [ ]:
frase = "<s> el gato come pescado y luego el gato duerme".split()
vocab = sorted(set(frase))
n, Vsz = len(frase), len(vocab)
E = np.eye(Vsz)[[vocab.index(w) for w in frase]]   # (n, |V|) one-hot de tokens
Pos = np.eye(n)                                    # (n, n) one-hot de posiciones
beta = 10.0

def dibujar(A, titulo, ax):
    ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(frase, rotation=90)
    ax.set_yticks(range(n)); ax.set_yticklabels(frase)
    ax.set_title(titulo); ax.set_xlabel("token mirado (key)")

**Ejercicio 5.** (a) Capa 1: construye `Q1 = beta * Pos` y `K1` como `beta` por `Pos` con las columnas desplazadas una posición (pista: `np.roll(Pos, 1, axis=1)` hace que la fila $j$ tenga el 1 en la columna $j+1$). Aplica `atencion_causal` con values `E` para obtener `H1, A1`. (b) Capa 2: `Q2 = beta * E`, `K2 = beta * H1`, values `E`; obtén `H2, A2`. (c) `prediccion` = lista con la palabra de `vocab` de mayor valor en cada fila de `H2`.

In [ ]:
# TU CÓDIGO
H1, A1 = None, None
H2, A2 = None, None
prediccion = None

In [ ]:
# Comprobación
assert A1 is not None and A1.shape == (n, n)
assert np.all(A1.argmax(axis=1)[1:] == np.arange(n - 1)), "La capa 1 debe mirar al token anterior (subdiagonal)"
i_el2, i_gato2 = 7, 8   # segunda aparición de "el" y de "gato"
assert prediccion[i_el2] == "gato", "Tras el segundo 'el' la cabeza de inducción debe predecir 'gato'"
assert prediccion[i_gato2] == "come", "Tras el segundo 'gato' debe predecir 'come'"
assert A2[i_el2, 2] > 0.99, "El segundo 'el' debe mirar a la posición 2 ('gato')"
print("Correcto:", list(zip(frase, prediccion))[6:])
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
dibujar(A1, "Capa 1: token anterior", axes[0])
dibujar(A2, "Capa 2: inducción", axes[1])
axes[0].set_ylabel("token que mira (query)")
plt.tight_layout(); plt.show()

**Pregunta.** En el mapa de la capa 2, la fila del segundo «el» brilla en la columna de «gato», no en la del primer «el». ¿Por qué? ¿Qué pasa en las filas de tokens que aparecen por primera vez?

Compara con una multi-cabeza **aleatoria** (sin entrenar): mapas difusos. Entrenar convierte ese ruido en patrones como los de arriba (módulo 15).

In [ ]:
rng = np.random.default_rng(5)
d, h = 16, 4
Xr = rng.standard_normal((n, d))
Ws = [rng.standard_normal((d, d)) for _ in range(4)]
_, Ar = multihead(Xr, *Ws, h=h)
fig, axes = plt.subplots(1, h, figsize=(14, 3.5))
for i in range(h):
    dibujar(Ar[i], f"Cabeza aleatoria {i}", axes[i])
plt.tight_layout(); plt.show()

## Parte 7 · El coste cuadrático, medido

La matriz $QK^\top$ tiene $n^2$ elementos. Si duplicas $n$, el tiempo tiende a multiplicarse por 4 cuando $n$ es grande (los tiempos pequeños son ruidosos).

In [ ]:
rng = np.random.default_rng(6)
dk = 64
ns = [512, 1024, 2048, 4096]
atencion_causal(*rng.standard_normal((3, 256, dk)))   # calentamiento
tiempos = []
for n_ in ns:
    Qb, Kb, Vb = rng.standard_normal((3, n_, dk))
    t0 = time.perf_counter()
    for _ in range(3):
        atencion_causal(Qb, Kb, Vb)
    tiempos.append((time.perf_counter() - t0) / 3)
for n_, t in zip(ns, tiempos):
    print(f"n = {n_:>5}: {1000 * t:7.1f} ms   memoria de A en fp16 = {n_**2 * 2 / 1e6:6.1f} MB")
print("Cociente de tiempos al duplicar n:", np.round(np.array(tiempos[1:]) / np.array(tiempos[:-1]), 1))

**Retos.**

1. Extrapola: ¿cuánto tardaría y cuánta memoria ocuparía tu implementación con $n = 100.000$? Compáralo con la tabla de la sección 13.6.
2. Implementa una máscara de **ventana deslizante**: el token $i$ solo mira a $j \in [i-w+1, i]$. ¿Cómo cambia el coste si solo calculas esas posiciones?
3. Cambia `beta` de la parte 6 por 1 y por 100. ¿Qué pasa con los mapas? Relaciónalo con la temperatura de la softmax.
4. Añade a la frase una tercera aparición de «el». ¿Qué predice la cabeza de inducción?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def atencion(Q, K, V):
#     S = Q @ K.T / np.sqrt(Q.shape[-1])
#     A = softmax(S)
#     return A @ V, A
#
# Ejercicio 2
# def varianza_producto(d_k, m, rng):
#     q = rng.standard_normal((m, d_k)); k = rng.standard_normal((m, d_k))
#     return (q * k).sum(axis=1).var()
#
# Ejercicio 3
# def mascara_causal(n):
#     return np.triu(np.full((n, n), -np.inf), k=1)
# def atencion_causal(Q, K, V):
#     n, d_k = Q.shape[-2], Q.shape[-1]
#     S = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k) + mascara_causal(n)
#     A = softmax(S)
#     return A @ V, A
#
# Ejercicio 4
# def dividir_cabezas(X, h):
#     n, d = X.shape
#     return X.reshape(n, h, d // h).transpose(1, 0, 2)
# def unir_cabezas(X):
#     h, n, d_k = X.shape
#     return X.transpose(1, 0, 2).reshape(n, h * d_k)
# def multihead(X, WQ, WK, WV, WO, h, causal=True):
#     Q, K, V = (dividir_cabezas(X @ W, h) for W in (WQ, WK, WV))
#     if causal:
#         O, A = atencion_causal(Q, K, V)
#     else:
#         A = softmax(Q @ np.swapaxes(K, -1, -2) / np.sqrt(Q.shape[-1])); O = A @ V
#     return unir_cabezas(O) @ WO, A
#
# Ejercicio 5
# H1, A1 = atencion_causal(beta * Pos, beta * np.roll(Pos, 1, axis=1), E)
# H2, A2 = atencion_causal(beta * E, beta * H1, E)
# prediccion = [vocab[k] for k in H2.argmax(axis=1)]